# Competitor Check Guardrail

Uses the **Competitor Check** validator from the [Guardrails Hub](https://guardrailsai.com/hub) to flag (or remove) mentions of competitor names in text.

Validators on the hub are now ordinary pip packages under the `guardrails_ai.*` namespace (the old `guardrails hub install hub://...` / `from guardrails.hub import ...` flow is no longer used).

**One-time setup** (already done in `myenv`):
```bash
pip install guardrails-ai guardrails-ai-competitor-check
python -m guardrails_ai.competitor_check.post_install   # downloads the NLTK punkt data + spaCy en_core_web_trf model
```

In [7]:
import warnings

# silence noisy (harmless) warnings from spaCy/torch and guardrails' sync fallback in Jupyter
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Could not obtain an event loop")

from dotenv import load_dotenv
from guardrails import Guard  ## the main wrapper for Guardrails
from guardrails.errors import ValidationError
from guardrails_ai.competitor_check import CompetitorCheck  ## new hub import path

# load the env vars
load_dotenv()

True

## 1. Create the validator and the guard

`Guard().use(...)` now takes **validator instances** — the old form `Guard().use(CompetitorCheck, competitor_list, on_fail="exception")` is no longer supported.

`configure(allow_metrics_collection=False)` turns off Guardrails' telemetry, which otherwise prints long `opentelemetry ... Max retries exceeded` tracebacks when it can't reach its endpoint.

In [8]:
competitor_list = ["Apple", "Google", "Microsoft", "Samsung", "Amazon"]

# create the validator
validator = CompetitorCheck(competitors=competitor_list, on_fail="exception")

# create guard and add the validator to the guard
guard = Guard().use(validator)
guard.configure(allow_metrics_collection=False)

print("Validator:", validator)
print("on_fail:", validator.on_fail_descriptor)

Validator: CompetitorCheck(rail_alias='guardrails/competitor_check')
on_fail: OnFailAction.EXCEPTION


### 2. Validate user input

The validator first does an exact (case-insensitive) match, then confirms with spaCy NER — so the fruit *apple* passes, while the company *Apple* is flagged.

In [9]:
user_input_1 = "I like all the fruits, but apple and grapes are my favorite."
user_input_2 = "Apple is great, makes durable and high quality products."

response_1 = guard.validate(user_input_1)
response_1

ValidationOutcome[~OT](call_id='2400852328128', raw_llm_output='I like all the fruits, but apple and grapes are my favorite.', validation_summaries=[], validated_output='I like all the fruits, but apple and grapes are my favorite.', reask=None, validation_passed=True, error=None)

In [10]:
print(f"Validation Result: {response_1.validation_passed}")
print(f"Raw User Input: {response_1.raw_llm_output}")
print(f"Validated Output: {response_1.validated_output}")

Validation Result: True
Raw User Input: I like all the fruits, but apple and grapes are my favorite.
Validated Output: I like all the fruits, but apple and grapes are my favorite.


In [11]:
# on_fail="exception" -> a competitor mention raises a ValidationError
try:
    response_2 = guard.validate(user_input_2)
except ValidationError as e:
    print(f"Blocked: {e}")

Blocked: Validation failed for field with errors: Found the following competitors: Apple. Please avoid naming those competitors next time


### 3. `on_fail="fix"` — redact instead of raising

With `fix`, the guard returns the text with competitor names replaced by `[COMPETITOR]`.

In [12]:
fix_guard = Guard().use(CompetitorCheck(competitors=competitor_list, on_fail="fix"))
fix_guard.configure(allow_metrics_collection=False)

response_3 = fix_guard.validate("Apple makes great phones. Samsung is good too. I love grapes.")

print(f"Validated Output: {response_3.validated_output}")
for summary in response_3.validation_summaries:
    print(f"Reason: {summary.failure_reason}")

Validated Output: [COMPETITOR] makes great phones.[COMPETITOR] is good too.I love grapes.
Reason: Found the following competitors: Apple, Samsung. Please avoid naming those competitors next time
